# BinarySearch (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/BinarySearch_java.ipynb)

Chapter 13 — Correctness. Runnable notebook of `BinarySearch`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/BinarySearch_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/BinarySearch_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the Java code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Binary search: BinarySearch

**Input:** an array $a[1..n]$ sorted in non-decreasing order and a value $m$. **Output:** an index $k$ with $a[k] = m$ if $m$ is in $a$, $-1$ otherwise. **Invariant:** (I1) $1 \le left$ and $right \le n$; (I2) if $m$ is in $a$, then $m$ is in $a[left..right]$.

Pseudo-code: section 13.4 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [2]:
%%writefile BinarySearch.java
public class BinarySearch
{
// "[a[from], ..., a[to-1]]"
static String fmt(int[] a, int from, int to)
{
    StringBuilder s = new StringBuilder("[");
    for (int i = from; i < to; i++)
    {
        if (i > from) s.append(", ");
        s.append(a[i]);
    }
    return s.append("]").toString();
}

static boolean equal(int[] x, int xs, int[] y, int ys, int len)     // x[xs..xs+len-1] == y[ys..ys+len-1]
{
    for (int k = 0; k < len; k++)
        if (x[xs + k] != y[ys + k]) return false;
    return true;
}

    // Invariant (I1) 1 <= left and right <= n, (I2) if m is in a then it is in a[left..right]; mid = -1: empty interval
    static void printRow(int[] a, int m, int left, int right, int mid)
    {
        int n = a.length;
        boolean inA = false, inSub = false;
        for (int k = 0; k < n; k++)
        {
            if (a[k] == m)
            {
                inA = true;
                if (k >= left && k <= right) inSub = true;
            }
        }
        boolean ok = (left + 1 >= 1 && right + 1 <= n && (!inA || inSub));
        String action;
        if (mid < 0)
            action = "left > right => return -1";
        else if (a[mid] == m)
            action = String.format("mid = %d, a[%d] = %d = %d => return %d", mid + 1, mid + 1, a[mid], m, mid + 1);
        else if (a[mid] < m)
            action = String.format("mid = %d, a[%d] = %d < %d => left = %d", mid + 1, mid + 1, a[mid], m, mid + 2);
        else
            action = String.format("mid = %d, a[%d] = %d > %d => right = %d", mid + 1, mid + 1, a[mid], m, mid);
        System.out.println(String.format("%4d | %5d | %-27s | %-11s | %s", left + 1, right + 1, fmt(a, left, right + 1), ok ? "ok" : "FALSE", action));
    }

    // BinarySearch(a, n, m)
    static int binarySearch(int[] a, int m, boolean trace)
    {
        int n = a.length;
        int left = 0;                              // left := 1
        int right = n - 1;                         // right := n
        while (left <= right)
        {
            int mid = left + (right - left) / 2;
            if (trace) printRow(a, m, left, right, mid);   // invariant row for this evaluation
            if (a[mid] == m)
            {
                return mid;
            }
            if (a[mid] < m)
            {
                left = mid + 1;
            }
            else
            {
                right = mid - 1;
            }
        }
        if (trace) printRow(a, m, left, right, -1);
        return -1;
    }

    public static void main(String[] args)
    {
        int[] a = {3, 9, 11, 12, 12, 16, 19};      // example of the lecture notes
        int n = a.length;
        System.out.printf("a = %s, n = %d\n", fmt(a, 0, n), n);

        int m = 10;
        System.out.printf("\n");
        System.out.printf("m = %d\n", m);
        System.out.printf("left | right | a[left..right]              | invariant   | action\n");
        int k = binarySearch(a, m, true);
        System.out.printf("m = %d not found: it returns %d\n", m, k);

        m = 12;
        System.out.printf("\n");
        System.out.printf("m = %d\n", m);
        System.out.printf("left | right | a[left..right]              | invariant   | action\n");
        k = binarySearch(a, m, true);
        System.out.printf("m = %d found: index %d in the code (position %d in the pseudo-code)\n", m, k, k + 1);

        // Exhaustive check for m = 0, ..., 20
        int errors = 0;
        for (m = 0; m <= 20; m++)
        {
            k = binarySearch(a, m, false);
            boolean inA = false;
            for (int t = 0; t < n; t++) if (a[t] == m) inA = true;
            if ((inA && (k < 0 || a[k] != m)) || (!inA && k != -1)) errors++;
        }
        System.out.printf("\n");
        System.out.printf("Check for m = 0, ..., 20: %d errors\n", errors);
    }
}

Writing BinarySearch.java


In [3]:
!javac BinarySearch.java && java BinarySearch

a = [3, 9, 11, 12, 12, 16, 19], n = 7

m = 10
left | right | a[left..right]              | invariant   | action
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 > 10 => right = 3
   1 |     3 | [3, 9, 11]                  | ok          | mid = 2, a[2] = 9 < 10 => left = 3
   3 |     3 | [11]                        | ok          | mid = 3, a[3] = 11 > 10 => right = 2
   3 |     2 | []                          | ok          | left > right => return -1
m = 10 not found: it returns -1

m = 12
left | right | a[left..right]              | invariant   | action
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 = 12 => return 4
m = 12 found: index 3 in the code (position 4 in the pseudo-code)

Check for m = 0, ..., 20: 0 errors


**Try it yourself.** Change the array (it must stay sorted) and the target value.

In [4]:
%%writefile ProvaBinarySearch.java
public class ProvaBinarySearch
{
    public static void main(String[] args)
    {
        int[] a = {2, 4, 4, 7, 9, 15, 21, 30};      // <-- change the array (sorted!)
        int m = 15;                                 // <-- and the target value
        System.out.printf("left | right | a[left..right]              | invariant   | action\n");
        int k = BinarySearch.binarySearch(a, m, true);
        System.out.printf("returns %d\n", k);
    }
}

Writing ProvaBinarySearch.java


In [5]:
!javac ProvaBinarySearch.java && java ProvaBinarySearch

left | right | a[left..right]              | invariant   | action
   1 |     8 | [2, 4, 4, 7, 9, 15, 21, 30] | ok          | mid = 4, a[4] = 7 < 15 => left = 5
   5 |     8 | [9, 15, 21, 30]             | ok          | mid = 6, a[6] = 15 = 15 => return 6
returns 5


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).